# Project Introduction

This notebook runs the Secure Retail Data Lakehouse project from top to bottom. It generates sample retail data, applies Bronze, Silver, and Gold layer processing, creates dashboards, and validates that sensitive PII and PCI data is protected.

## Problem Statement

Retail platforms collect names, addresses, phone numbers, dates of birth, Aadhaar numbers, card numbers, card expiry values, and CVV data. Plain-text storage creates security, fraud, and compliance risks. The lakehouse must support analytics while masking, tokenizing, aggregating, and hard-dropping sensitive data.

## Objectives

- Ingest retail data into a Bronze layer with minimal validation.
- Hard-drop CVV immediately during ingestion.
- Mask PII and PCI fields in the Silver layer.
- Tokenize customers with SHA-256/HMAC.
- Engineer age bands and spend categories.
- Build Gold layer aggregates and dashboards.
- Validate that protected outputs do not expose sensitive data.

## Tech Stack

- Python
- pandas
- matplotlib
- Jupyter Notebook
- CSV-based batch lakehouse layers

## Project Architecture

Raw data is generated into `data/`, ingested into `bronze/`, transformed into masked and tokenized records in `silver/`, aggregated into `gold/`, and visualized in `dashboard/`.

## Imports

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from generate_sample_data import main as generate_sample_data
from pipeline import (
    RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_DIR,
    ingest_bronze, build_silver, build_gold, validate_layers,
    mask_name, mask_email, mask_phone, mask_aadhaar, mask_card,
)
from dashboard import main as generate_dashboard

pd.set_option('display.max_columns', 40)

## Directory Creation

In [2]:
for folder in ['data', 'bronze', 'silver', 'gold', 'dashboard']:
    (ROOT / folder).mkdir(exist_ok=True)
print('Project directories are ready.')

Project directories are ready.


## Load Data

In [3]:
generate_sample_data()
raw_df = pd.read_csv(RAW_PATH)
print('Raw Dataset')
display(raw_df.head())

Wrote 1000 raw transactions to C:\Users\eklav\OneDrive\Documents\Secure Retail Lakehouse\data\retail_transactions.csv
Raw Dataset


,transaction_id,transaction_ts,customer_id,full_name,email,phone,date_of_birth,address,city,region,aadhaar_number,card_number,card_expiry,card_cvv,payment_method,channel,product_category,amount
0,TXN000001,2025-12-09T11:32,CUST0099,Aditya Sharma,aditya.sharma99@example.com,9312332805,1966-11-25,"838 Park Street, Delhi",Delhi,North,151070296670,4662532447209993,11/29,531,Net Banking,E-Commerce,Sports,1679.34
1,TXN000002,2025-05-08T16:26,CUST0056,Riya Patel,riya.patel56@example.com,9759737342,1974-05-24,"524 Park Street, Chennai",Chennai,South,992299590010,4943969078447364,08/27,354,Wallet,E-Commerce,Sports,2214.29
2,TXN000003,2025-01-06T21:57,CUST0108,Aditya Mishra,aditya.mishra108@example.com,9316809172,1955-08-24,"161 Ring Road, Kolkata",Kolkata,East,511686854870,4998616229991191,05/31,956,Debit Card,Point-of-Sale,Beauty,2200.81
3,TXN000004,2025-03-26T02:33,CUST0110,Priya Das,priya.das110@example.com,9301794412,1980-03-24,"945 Lake View, Delhi",Delhi,North,785082148628,4902330307053457,09/30,769,Wallet,Point-of-Sale,Home,2859.60
4,TXN000005,2025-04-25T18:47,CUST0031,Vikram Mehta,vikram.mehta31@example.com,9282518553,1997-05-31,"634 Lake View, Bengaluru",Bengaluru,South,937923747407,4482175946474367,02/28,658,Wallet,Point-of-Sale,Fashion,2443.17


## Bronze Layer

The Bronze layer keeps the ingested records with minimal validation. CVV is hard-dropped immediately because it is highly sensitive PCI data and should not persist beyond ingestion.

In [4]:
bronze_df = ingest_bronze()
print('Bronze Dataset')
display(bronze_df.head())
print('CVV present in Bronze:', 'card_cvv' in bronze_df.columns)

Bronze Dataset


,transaction_id,transaction_ts,customer_id,full_name,email,phone,date_of_birth,address,city,region,aadhaar_number,card_number,card_expiry,payment_method,channel,product_category,amount
0,TXN000001,2025-12-09T11:32,CUST0099,Aditya Sharma,aditya.sharma99@example.com,9312332805,1966-11-25,"838 Park Street, Delhi",Delhi,North,151070296670,4662532447209993,11/29,Net Banking,E-Commerce,Sports,1679.34
1,TXN000002,2025-05-08T16:26,CUST0056,Riya Patel,riya.patel56@example.com,9759737342,1974-05-24,"524 Park Street, Chennai",Chennai,South,992299590010,4943969078447364,08/27,Wallet,E-Commerce,Sports,2214.29
2,TXN000003,2025-01-06T21:57,CUST0108,Aditya Mishra,aditya.mishra108@example.com,9316809172,1955-08-24,"161 Ring Road, Kolkata",Kolkata,East,511686854870,4998616229991191,05/31,Debit Card,Point-of-Sale,Beauty,2200.81
3,TXN000004,2025-03-26T02:33,CUST0110,Priya Das,priya.das110@example.com,9301794412,1980-03-24,"945 Lake View, Delhi",Delhi,North,785082148628,4902330307053457,09/30,Wallet,Point-of-Sale,Home,2859.60
4,TXN000005,2025-04-25T18:47,CUST0031,Vikram Mehta,vikram.mehta31@example.com,9282518553,1997-05-31,"634 Lake View, Bengaluru",Bengaluru,South,937923747407,4482175946474367,02/28,Wallet,Point-of-Sale,Fashion,2443.17


CVV present in Bronze: False


## Silver Layer

In [5]:
silver_df = build_silver(bronze_df)
print('Silver Dataset')
display(silver_df.head())

Silver Dataset


,transaction_id,transaction_ts,city,region,payment_method,channel,product_category,amount,customer_token,name_masked,email_masked,phone_masked,aadhaar_masked,card_masked,age_band,spend_category,transaction_month
0,TXN000001,2025-12-09T11:32,Delhi,North,Net Banking,E-Commerce,Sports,1679.34,e0af8fe1368b95aa1caacb48,A*** S***,ad***@example.com,******2805,XXXX-XXXX-6670,**** **** **** 9993,55-64,Medium (1000-5000),2025-12
1,TXN000002,2025-05-08T16:26,Chennai,South,Wallet,E-Commerce,Sports,2214.29,c5818ae95f3cda87590d785a,R*** P***,ri***@example.com,******7342,XXXX-XXXX-0010,**** **** **** 7364,45-54,Medium (1000-5000),2025-05
2,TXN000003,2025-01-06T21:57,Kolkata,East,Debit Card,Point-of-Sale,Beauty,2200.81,eb273d12b231d6bcbb09c24b,A*** M***,ad***@example.com,******9172,XXXX-XXXX-4870,**** **** **** 1191,65+,Medium (1000-5000),2025-01
3,TXN000004,2025-03-26T02:33,Delhi,North,Wallet,Point-of-Sale,Home,2859.60,c48ec84a8d2bc08ed04c9d39,P*** D***,pr***@example.com,******4412,XXXX-XXXX-8628,**** **** **** 3457,45-54,Medium (1000-5000),2025-03
4,TXN000005,2025-04-25T18:47,Bengaluru,South,Wallet,Point-of-Sale,Fashion,2443.17,77cb6da744b1ded08acd60e9,V*** M***,vi***@example.com,******8553,XXXX-XXXX-7407,**** **** **** 4367,25-34,Medium (1000-5000),2025-04


In [6]:
masking_examples = pd.DataFrame({
    'OriginalName': raw_df['full_name'].head(5),
    'MaskedName': raw_df['full_name'].head(5).map(mask_name),
    'OriginalEmail': raw_df['email'].head(5),
    'MaskedEmail': raw_df['email'].head(5).map(mask_email),
    'OriginalPhone': raw_df['phone'].head(5),
    'MaskedPhone': raw_df['phone'].head(5).map(mask_phone),
    'OriginalAadhaar': raw_df['aadhaar_number'].head(5),
    'MaskedAadhaar': raw_df['aadhaar_number'].head(5).map(mask_aadhaar),
    'OriginalCard': raw_df['card_number'].head(5),
    'MaskedCard': raw_df['card_number'].head(5).map(mask_card),
})
print('Before-and-after masking examples')
display(masking_examples)

Before-and-after masking examples


,OriginalName,MaskedName,OriginalEmail,MaskedEmail,OriginalPhone,MaskedPhone,OriginalAadhaar,MaskedAadhaar,OriginalCard,MaskedCard
0,Aditya Sharma,A*** S***,aditya.sharma99@example.com,ad***@example.com,9312332805,******2805,151070296670,XXXX-XXXX-6670,4662532447209993,**** **** **** 9993
1,Riya Patel,R*** P***,riya.patel56@example.com,ri***@example.com,9759737342,******7342,992299590010,XXXX-XXXX-0010,4943969078447364,**** **** **** 7364
2,Aditya Mishra,A*** M***,aditya.mishra108@example.com,ad***@example.com,9316809172,******9172,511686854870,XXXX-XXXX-4870,4998616229991191,**** **** **** 1191
3,Priya Das,P*** D***,priya.das110@example.com,pr***@example.com,9301794412,******4412,785082148628,XXXX-XXXX-8628,4902330307053457,**** **** **** 3457
4,Vikram Mehta,V*** M***,vikram.mehta31@example.com,vi***@example.com,9282518553,******8553,937923747407,XXXX-XXXX-7407,4482175946474367,**** **** **** 4367


## Gold Layer

In [7]:
gold_outputs = build_gold(silver_df)
print('Gold Dataset: Customer Spend')
display(gold_outputs['customer_spend'].head())
print('Gold Dataset: Spend Category Summary')
display(gold_outputs['spend_category_summary'])
print('Gold Dataset: Payment Method Distribution')
display(gold_outputs['payment_method_distribution'])

Gold Dataset: Customer Spend


,customer_token,total_spend,avg_spend,transactions
58,7ead03eda83c3cc44cd1e0ea,72830.78,7283.078000,10
18,2ce30ecdf81f4c29061c1008,65456.29,5454.690833,12
49,71babf2c38d3a6da3a60a693,61686.29,4406.163571,14
8,1b1e5b2abc78a9a541cb94a2,58334.73,4487.286923,13
2,08b8dad8d18195ab647ec9a0,58232.44,8318.920000,7


Gold Dataset: Spend Category Summary


,spend_category,total_amount,avg_amount,transactions
0,High (>5000),2150719.92,10917.360000,197
2,Medium (1000-5000),1616892.97,2546.288142,635
1,Low (<1000),113923.59,678.116607,168


Gold Dataset: Payment Method Distribution


,payment_method,total_amount,transactions
4,Wallet,802092.95,211
1,Debit Card,712754.87,204
0,Credit Card,752313.59,196
2,Net Banking,844128.98,196
3,UPI,770246.09,193


## Dashboard Generation

In [8]:
generate_dashboard()
print('Dashboard HTML:', ROOT / 'dashboard' / 'dashboard.html')
print('Dashboard PNG:', ROOT / 'dashboard' / 'dashboard.png')

Wrote HTML dashboard to C:\Users\eklav\OneDrive\Documents\Secure Retail Lakehouse\dashboard\dashboard.html
Wrote static dashboard image to C:\Users\eklav\OneDrive\Documents\Secure Retail Lakehouse\dashboard\dashboard.png
Dashboard HTML: c:\Users\eklav\OneDrive\Documents\Secure Retail Lakehouse\dashboard\dashboard.html
Dashboard PNG: c:\Users\eklav\OneDrive\Documents\Secure Retail Lakehouse\dashboard\dashboard.png


## Validation Checks

In [9]:
validate_layers(verbose=True)

SUCCESS: CVV column does not exist in Bronze.
SUCCESS: Card numbers are masked.
SUCCESS: Emails are masked.
SUCCESS: Phone numbers are masked.
SUCCESS: CustomerToken exists.
SUCCESS: AgeBand exists.
SUCCESS: SpendCategory exists.


## Conclusion

The project successfully implements a secure retail lakehouse pipeline. CVV is hard-dropped during Bronze ingestion, direct identifiers are masked or tokenized in Silver, and Gold outputs provide safe business analytics for dashboards and reporting.